In [5]:
import csv
import json

# Step 1: Open and read CSV file
with open("C:\\Users\\Dipali\\OneDrive\\Documents\\CareLopp_Clinic_Project\\appointments.csv", "r", newline="") as file:

    reader = csv.DictReader(file)

    # Print column headers
    print("Headers:", reader.fieldnames)

    # Read all rows
    rows = list(reader)

    # Print total number of rows
    print("Total rows:", len(rows))


# Step 2: Create a function to convert data types
def parse_row(row):

    row["age"] = int(row["age"])

    row["fee_inr"] = int(row["fee_inr"])

    row["prior_no_shows"] = int(row["prior_no_shows"])

    row["distance_km"] = float(row["distance_km"])

    return row


# Step 3: Create accepted and rejected lists
accepted = []
rejected = []


# Step 4: Process every row
for row in rows:

    try:
        converted_row = parse_row(row)

        accepted.append(converted_row)

    except ValueError:
        rejected.append(row)


# Step 5: Print results
print("Accepted rows:", len(accepted))
print("Rejected rows:", len(rejected))


# Step 6: Save rejected rows into JSON file
with open("rejected_rows.json", "w") as file:

    json.dump(rejected, file, indent=2)

print("Rejected rows saved successfully!")


Headers: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total rows: 1020
Accepted rows: 988
Rejected rows: 32
Rejected rows saved successfully!


In [80]:
import pandas as pd

# STEP 1: Load the original CSV file
df = pd.read_csv(
    r"C:\\Users\\Dipali\\OneDrive\\Documents\\CareLopp_Clinic_Project\\appointments.csv"
)

# STEP 2: Inspect the original dataset

print("Dataset Shape:")
print(df.shape)

print("\nDataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())


# STEP 3: Remove exact duplicate rows
df = df.drop_duplicates()

print("\nAfter removing duplicates:", len(df))


# STEP 4: Clean booking_channel
df["booking_channel"] = (
    df["booking_channel"].str.strip().str.title()
)

print("After cleaning booking_channel:", len(df))


# STEP 5: Convert appt_date into datetime
df["appt_date"] = pd.to_datetime(
    df["appt_date"],
    errors="coerce"
)

# Convert booked_on into datetime
df["booked_on"] = pd.to_datetime(df["booked_on"])

print("After date conversion:", len(df))


# STEP 6: Drop rows with missing appt_date or status
df = df.dropna(subset=["appt_date", "status"])

print("After dropping missing dates/status:", len(df))


# STEP 7: Display final cleaned dataset
print("\nFinal Dataset Shape:")
print(df.shape)

print("\nFinal Dataset:")
print(df.head())


Dataset Shape:
(1020, 14)

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing Values:
appointment_id      0
patie

C:\Users\Dipali\AppData\Local\Temp\ipykernel_23568\788379958.py:38: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["appt_date"] = pd.to_datetime(
C:\Users\Dipali\AppData\Local\Temp\ipykernel_23568\788379958.py:44: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["booked_on"] = pd.to_datetime(df["booked_on"])


In [81]:
import numpy as np

# STEP A: Treat invalid ages as missing values
df.loc[
    (df["age"] < 1) | (df["age"] > 100),
    "age"
] = np.nan


# STEP B: Fill missing age with median
df["age"] = df["age"].fillna(df["age"].median())

# Round age and convert to integer
df["age"] = df["age"].round().astype(int)


# Fill missing distance_km with median
df["distance_km"] = df["distance_km"].fillna(
    df["distance_km"].median()
)


# STEP C: Detect and cap outliers in distance_km

Q1 = df["distance_km"].quantile(0.25)
Q3 = df["distance_km"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# Count distance outliers before capping
distance_capped = (
    (df["distance_km"] < lower) |
    (df["distance_km"] > upper)
).sum()

# Cap distance outliers
df["distance_km"] = np.clip(
    df["distance_km"], lower, upper
)

print("Distance values capped:", distance_capped)


# Detect and cap outliers in fee_inr

Q1 = df["fee_inr"].quantile(0.25)
Q3 = df["fee_inr"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# Count fee outliers before capping
fee_capped = (
    (df["fee_inr"] < lower) |
    (df["fee_inr"] > upper)
).sum()

# Cap fee outliers
df["fee_inr"] = np.clip(
    df["fee_inr"], lower, upper
)

print("Fee values capped:", fee_capped)


# STEP D: Round distance_km to 1 decimal
df["distance_km"] = df["distance_km"].round(1)


# Round fee_inr to whole rupees
df["fee_inr"] = df["fee_inr"].round().astype(int)


# Display final results
print("\nFinal Dataset:")
print(df.head())

print("\nFinal Data Types:")
print(df[["age", "distance_km", "fee_inr"]].dtypes)


Distance values capped: 26
Fee values capped: 3

Final Dataset:
  appointment_id patient_id clinic_id  booked_on  appt_date  age gender  \
0          A0001       P509       C05 2025-01-24 2025-01-25   60      F   
1          A0002       P046       C07 2025-05-27 2025-06-06   39      M   
2          A0003       P009       C02 2025-03-24 2025-03-30   63      F   
4          A0004       P482       C01 2025-04-18 2025-04-22   52      F   
5          A0005       P251       C01 2024-11-23 2025-01-02   64      F   

   distance_km  fee_inr booking_channel    visit_type reminder_sent  \
0         18.0      850           Phone  Consultation           Yes   
1         12.5      350         Walk-In     Follow-up            No   
2          4.1      850         Walk-In  Consultation            No   
4          7.5      750             App  Consultation           Yes   
5         19.0      700           Phone  Consultation            No   

   prior_no_shows    status  
0               1   No-Show 

In [82]:

# ============================================================
# TASK 1.4 — FEATURES AND NUMPY SUMMARY
# ============================================================

# STEP 1: Import NumPy
import numpy as np

# STEP 2: Create lead_days column
# Calculate days between booked_on and appt_date

df["lead_days"] = (
    df["appt_date"] - df["booked_on"]
).dt.days


# STEP 3: Create no_show column
# If status is No-Show, assign 1; otherwise assign 0

df["no_show"] = (
    df["status"] == "No-Show"
).astype(int)


# STEP 4: Select four columns and convert into NumPy array

X = df[
    ["age", "distance_km", "fee_inr", "lead_days"]
].to_numpy()


# STEP 5: Calculate mean of each column
# axis=0 calculates column-wise mean

column_means = np.mean(X, axis=0)

print("Mean of each column:", column_means)


# STEP 6: Find appointments where lead_days > 14

mask = df["lead_days"] > 14

# Count the number of True values
count = mask.sum()

# Calculate the share of True values
share = mask.mean()

print("Appointments with lead_days > 14:", count)
print("Share of appointments:", share)


# STEP 7: Calculate z-scores for distance_km

x = df["distance_km"].to_numpy()

z = (x - np.mean(x)) / np.std(x)

# Count rows where absolute z-score > 2
outlier_count = np.sum(np.abs(z) > 2)

print("Distance z-score outliers:", outlier_count)


# STEP 8: Display updated dataset

print("\nUpdated Dataset:")
display(df.head())


Mean of each column: [ 48.17373737   9.50090909 635.71414141  11.81919192]
Appointments with lead_days > 14: 296
Share of appointments: 0.298989898989899
Distance z-score outliers: 53

Updated Dataset:


,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
4,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
5,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1


In [84]:

# ============================================================
# TASK 1.5 - ENRICH THE CLINICS USING OPEN-METEO API
# ============================================================


# STEP 1: IMPORT REQUIRED LIBRARIES

import requests
import time


# STEP 2: LOAD THE CLINICS DATASET

# Replace clinics.csv with your actual clinics CSV filename
clinics = pd.read_csv("C:\\Users\\Dipali\\OneDrive\\Documents\\CareLopp_Clinic_Project\\clinics.csv")

# Display first 5 rows
print(clinics.head())

# Display all column names
print(clinics.columns.tolist())


# STEP 3: DEFINE THE REQUIRED COLUMN NAMES

clinic_name_col = "clinic_name"
latitude_col = "latitude"
longitude_col = "longitude"

# Check whether all required columns are available
required_columns = [
    clinic_name_col,
    latitude_col,
    longitude_col
]

for column in required_columns:
    if column not in clinics.columns:
        raise KeyError(f"Missing required column: {column}")

print("Required columns verified successfully.")


# STEP 4: CREATE A FUNCTION TO FETCH TEMPERATURE

def get_temperature(latitude, longitude, clinic_name):

    # STEP 4.1: DEFINE THE OPEN-METEO API URL

    url = "https://api.open-meteo.com/v1/forecast"


    # STEP 4.2: DEFINE API PARAMETERS

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m",
        "forecast_days": 1
    }


    # STEP 4.3: SET TWO ATTEMPTS (FIRST TRY + ONE RETRY)

    for attempt in range(2):

        try:

            # STEP 4.4: SEND API REQUEST WITH 10 SECONDS TIMEOUT

            response = requests.get(
                url,
                params=params,
                timeout=10
            )


            # STEP 4.5: CHECK THE API STATUS CODE

            if response.status_code == 200:

                # STEP 4.6: READ THE API RESPONSE IN JSON FORMAT

                data = response.json()


                # STEP 4.7: EXTRACT HOURLY TEMPERATURE DATA

                hourly_data = data.get("hourly", {})

                temperatures = hourly_data.get("temperature_2m")


                # STEP 4.8: CHECK WHETHER TEMPERATURE DATA EXISTS

                if temperatures and temperatures[0] is not None:

                    # STEP 4.9: TAKE THE FIRST TEMPERATURE VALUE

                    temp_c = temperatures[0]

                    return temp_c

                else:

                    reason = "Hourly temperature data is missing"

            else:

                # STEP 4.10: HANDLE NON-200 STATUS CODE

                reason = f"API returned status code {response.status_code}"


        # STEP 4.11: HANDLE API REQUEST ERRORS

        except requests.exceptions.RequestException as error:

            reason = str(error)


        # STEP 4.12: RETRY THE CLINIC ONCE IF THE FIRST ATTEMPT FAILS

        if attempt == 0:

            print(f"{clinic_name}: Request failed. Retrying once...")

            time.sleep(1)


    # STEP 4.13: RETURN NONE IF BOTH ATTEMPTS FAIL

    print(f"{clinic_name}: Temperature unavailable. Reason: {reason}")

    return None


# STEP 5: CREATE A NEW TEMPERATURE COLUMN

clinics["temp_c"] = None


# STEP 6: LOOP THROUGH ALL 8 CLINICS

for index, row in clinics.iterrows():

    # STEP 6.1: GET CLINIC NAME

    clinic_name = row[clinic_name_col]


    # STEP 6.2: GET CLINIC LATITUDE

    latitude = row[latitude_col]


    # STEP 6.3: GET CLINIC LONGITUDE

    longitude = row[longitude_col]


    # STEP 6.4: CALL THE TEMPERATURE FUNCTION

    temp_c = get_temperature(
        latitude,
        longitude,
        clinic_name
    )


    # STEP 6.5: STORE TEMPERATURE IN THE DATAFRAME

    clinics.loc[index, "temp_c"] = temp_c


    # STEP 6.6: PRINT THE TEMPERATURE RESULT

    if temp_c is not None:

        print(f"{clinic_name}: Temperature = {temp_c} °C")


# STEP 7: CHECK THE ENRICHED DATASET

print("\nEnriched Clinics Dataset:")

print(clinics.head(8))


# STEP 8: CHECK MISSING TEMPERATURE VALUES

print("\nMissing Temperature Values:")

print(clinics["temp_c"].isnull().sum())

  clinic_id             clinic_name    city    tier  latitude  longitude
0       C01   Kothrud Family Clinic    Pune   Metro   18.5074    73.8077
1       C02    Hadapsar Care Centre    Pune   Metro   18.5089    73.9260
2       C03  Sitabuldi Health Point  Nagpur  Tier-2   21.1458    79.0882
3       C04    Malviya Nagar Clinic  Jaipur  Tier-2   26.8553    75.8150
4       C05   Vaishali Nagar Clinic  Jaipur  Tier-2   26.9124    75.7873
['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude']
Required columns verified successfully.
Kothrud Family Clinic: Temperature = 19.4 °C
Hadapsar Care Centre: Temperature = 19.7 °C
Sitabuldi Health Point: Temperature = 19.7 °C
Malviya Nagar Clinic: Temperature = 16.4 °C
Vaishali Nagar Clinic: Temperature = 16.4 °C
Edappally Medical Centre: Temperature = 26.3 °C
Vijay Nagar Clinic: Temperature = 16.1 °C
Gomti Nagar Care Point: Temperature = 14.7 °C

Enriched Clinics Dataset:
  clinic_id               clinic_name     city    tier  latitude 

In [92]:

# ============================================================
# TASK 1.6 — EXPORT THE CLEAN TABLE AND WRITE SQL LOAD SCRIPT
# ============================================================


# STEP 1: Import required libraries

import pandas as pd
import numpy as np
import os


# STEP 2: Define the exact 16 required appointment columns

columns = [
    "appointment_id",
    "patient_id",
    "clinic_id",
    "booked_on",
    "appt_date",
    "age",
    "gender",
    "distance_km",
    "fee_inr",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "prior_no_shows",
    "status",
    "lead_days",
    "no_show"
]


# STEP 3: Check whether all required columns exist

missing_columns = [
    col for col in columns if col not in df.columns
]

assert len(missing_columns) == 0, \
    f"Missing columns: {missing_columns}"


# STEP 4: Select the required columns

appointments = df[columns].copy()


# STEP 5: Check appointment_id uniqueness

print("Are appointment IDs unique?",
      appointments["appointment_id"].is_unique)


# STEP 6: Check missing values in all 16 columns

print("\nMissing values in each column:")
print(appointments.isnull().sum())

total_missing = appointments.isnull().sum().sum()

print("\nTotal missing values:", total_missing)


# STEP 7: Validate appointment data

assert appointments["appointment_id"].is_unique, \
    "Error: Duplicate appointment IDs found!"

assert total_missing == 0, \
    "Error: Missing values found in appointments data!"


# STEP 8: Sort appointments by appointment_id

appointments = appointments.sort_values(
    by="appointment_id"
).reset_index(drop=True)


# STEP 9: Format booked_on as YYYY-MM-DD

appointments["booked_on"] = pd.to_datetime(
    appointments["booked_on"]
).dt.strftime("%Y-%m-%d")


# STEP 10: Format appt_date as YYYY-MM-DD

appointments["appt_date"] = pd.to_datetime(
    appointments["appt_date"]
).dt.strftime("%Y-%m-%d")


# STEP 11: Select exactly 16 columns in the required order

appointments = appointments[columns]


# STEP 12: Save appointments_clean.csv

appointments.to_csv(
    "appointments_clean.csv",
    index=False
)

print("\nappointments_clean.csv created successfully!")


# STEP 13: Verify appointments_clean.csv

appointments_check = pd.read_csv(
    "appointments_clean.csv"
)

print("\nAppointment CSV verification:")
print("Total rows:", len(appointments_check))
print("Total columns:", len(appointments_check.columns))

print("\nColumn names:")
print(appointments_check.columns.tolist())

print("\nFirst 5 records:")
display(appointments_check.head())


# ============================================================
# STEP 14: READ ENRICHED CLINICS DATA FROM TASK 1.5
# ============================================================

clinics = pd.read_csv("clinics_enriched.csv")

print("\nClinic column names:")
print(clinics.columns.tolist())

print("\nClinic records:")
display(clinics.head())

print("\nTotal clinics:", len(clinics))


# ============================================================
# STEP 15: CREATE FUNCTION TO CONVERT PYTHON VALUES INTO SQL
# ============================================================

def sql_value(value):

    # Handle missing values
    if pd.isna(value):
        return "NULL"

    # Handle Boolean values
    if isinstance(value, (bool, np.bool_)):
        return "1" if value else "0"

    # Handle numeric values
    if isinstance(value, (int, float, np.integer, np.floating)):
        return str(value)

    # Handle text values
    return "'" + str(value).replace("'", "''") + "'"


# ============================================================
# STEP 16: CREATE load_data.sql FILE
# ============================================================

with open("load_data.sql", "w", encoding="utf-8") as file:

    # Write SQL file heading
    file.write("-- CareLopp Clinic Data Loading Script\n\n")

    # --------------------------------------------------------
    # STEP 17: WRITE CLINIC INSERT STATEMENTS FIRST
    # --------------------------------------------------------

    file.write("-- Insert clinic records first\n")

    # Loop through each clinic row
    for row in clinics.itertuples(index=False, name=None):

        # Convert values into SQL format
        values = ", ".join(
            sql_value(value) for value in row
        )

        # Create INSERT statement using f-string
        sql = f"INSERT INTO clinics VALUES ({values});\n"

        # Write SQL statement into file
        file.write(sql)

    # Add blank line
    file.write("\n")

    # --------------------------------------------------------
    # STEP 18: WRITE APPOINTMENT INSERT STATEMENTS SECOND
    # --------------------------------------------------------

    file.write("-- Insert appointment records after clinics\n")

    # Loop through each appointment row
    for row in appointments.itertuples(index=False, name=None):

        # Convert values into SQL format
        values = ", ".join(
            sql_value(value) for value in row
        )

        # Create INSERT statement using f-string
        sql = f"INSERT INTO appointments VALUES ({values});\n"

        # Write SQL statement into file
        file.write(sql)


# ============================================================
# STEP 19: VERIFY load_data.sql
# ============================================================

with open("load_data.sql", "r", encoding="utf-8") as file:
    sql_content = file.readlines()


# Count clinic INSERT statements
clinic_count = sum(
    line.startswith("INSERT INTO clinics")
    for line in sql_content
)


# Count appointment INSERT statements
appointment_count = sum(
    line.startswith("INSERT INTO appointments")
    for line in sql_content
)


print("\n========== SQL FILE VERIFICATION ==========")

print("Clinic INSERT statements:", clinic_count)

print("Appointment INSERT statements:", appointment_count)

print(
    "Total INSERT statements:",
    clinic_count + appointment_count
)


# Display first 10 SQL statements
print("\nFirst 10 lines of SQL file:")
print("".join(sql_content[:10]))


# ============================================================
# STEP 20: DISPLAY FINAL FILE LOCATIONS
# ============================================================

print("\n========== TASK 1.6 COMPLETED ==========")

print(
    "appointments_clean.csv:",
    os.path.abspath("appointments_clean.csv")
)

print(
    "load_data.sql:",
    os.path.abspath("load_data.sql")
)


Are appointment IDs unique? True

Missing values in each column:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64

Total missing values: 0

appointments_clean.csv created successfully!

Appointment CSV verification:
Total rows: 990
Total columns: 16

Column names:
['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show']

First 5 records:


,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
3,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
4,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1



Clinic column names:
['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude', 'temp_c']

Clinic records:


,clinic_id,clinic_name,city,tier,latitude,longitude,temp_c
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077,19.4
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260,19.7
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882,19.7
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150,16.4
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873,16.4



Total clinics: 8

========== SQL FILE VERIFICATION ==========
Clinic INSERT statements: 8
Appointment INSERT statements: 990
Total INSERT statements: 998

First 10 lines of SQL file:
-- CareLopp Clinic Data Loading Script

-- Insert clinic records first
INSERT INTO clinics VALUES ('C01', 'Kothrud Family Clinic', 'Pune', 'Metro', 18.5074, 73.8077, 19.4);
INSERT INTO clinics VALUES ('C02', 'Hadapsar Care Centre', 'Pune', 'Metro', 18.5089, 73.926, 19.7);
INSERT INTO clinics VALUES ('C03', 'Sitabuldi Health Point', 'Nagpur', 'Tier-2', 21.1458, 79.0882, 19.7);
INSERT INTO clinics VALUES ('C04', 'Malviya Nagar Clinic', 'Jaipur', 'Tier-2', 26.8553, 75.815, 16.4);
INSERT INTO clinics VALUES ('C05', 'Vaishali Nagar Clinic', 'Jaipur', 'Tier-2', 26.9124, 75.7873, 16.4);
INSERT INTO clinics VALUES ('C06', 'Edappally Medical Centre', 'Kochi', 'Tier-2', 10.0261, 76.3085, 26.3);
INSERT INTO clinics VALUES ('C07', 'Vijay Nagar Clinic', 'Indore', 'Tier-2', 22.7533, 75.8937, 16.1);


========== TASK 1.